# Cifrado de Vernam/OTP

## Implementacion sencilla, no segura

Tenemos un mensaje m con una cantidad de bits $|M|$, una llave k con una cantidad de bits $|K|$. Ademas, hay un esquema de encripcion denotado $e$. Por lema de Shannon, la cantidad $|K|\geq |M|$

Por ende, tiene que haber la misma cantidad de bits de llave que de mensaje. Para el OTP, el esquema de encripcion sera la compuerta XOR:

$$
\text{Cifrado: } e= m\oplus k
$$

$$
\text{Descifrado: } m= e\oplus k
$$
Vamos a encriptar. En criptografia es estandar convertir cada letra de un string a Unicode/ASCII

In [9]:
def otp_esquema(m,k):
    """
    Toma el mensaje, y por letra del mismo, lo convierte a ASCII, aplica la compuerta
    XOR, convierte devuelta de ASCII a caracter, y finalmente agrega dicha letra al string,
    iterando hasta completar el mensaje
    """
    mensaje_encriptado = ""
    for i in range(len(m)):
        #chr y el ord corresponden al tipo caracter y el tipo byte
        char = chr( ord(m[i]) ^ ord(k[i]) )
        mensaje_encriptado += char #corresponde a e
    return mensaje_encriptado
    

#Plaintext y llave

mensaje = "Hello" #Voy a escoger arbitrariamente el mensaje
llave = "World" #y la llave

mensaje_encriptado = otp_esquema(mensaje,llave)
#repr es opcional, es para visualizar el hecho que los bytes en UNICODE
#Que no solo corresponden a letras, pueden corresponder a tabulaciones o similar
print(f'Mensaje encriptado: ', repr(mensaje_encriptado))

#Mañosamente, nootamos que el cifrado es simetrico, entonces no hace falta hacer otra funcion

mensaje_descifrado = otp_esquema(mensaje_encriptado,llave)

print(f'Mensaje descifrado: ', mensaje_descifrado)



Mensaje encriptado:  '\x1f\n\x1e\x00\x0b'
Mensaje descifrado:  Hello


¿Que problemas tiene esto?

1. La llave está fija
2. Tenemos que generar una llave aleatoria, pero no podemos generarla normalmente
3. Hay que desechar la llave cada vez que encriptemos
4. Optimizar

Empezaremos por la llave, usaremos una libreria para generar numeros aleatorios criptograficamente seguros.


In [10]:
import secrets
#Tambien está os, pero es peor

Creamos la funcion para generar la clave.

In [11]:
def generar_clave(longitud_bits):
    """
    Genera una clave como una lista de bits.
    Fisicamente analogo a tirar una moneda N veces
    """
    clave = [secrets.choice([0,1]) for bit in range(longitud_bits)]
    return clave

Ahora, vamos a hacer el esquema de encripcion

1. El criptograma será manejado en bytes (valores entre 0 y 255)
2. Se usara la funcion zip, que toma dos listas y crea una lista de tuplas

In [3]:
def otp(mensaje_bytes,llave_bytes):
    """
    Aplica XOR bit por bit directamente sobre los arreglos de bytes
    """
    criptograma = bytes(m ^ c for m, c in zip(mensaje_bytes,llave_bytes))
    return criptograma

Solo por estética, vamos a convertir de bytes a bits

In [4]:
def bytes_a_bits(datos_bytes):
    """
    Toma cada paquete de 8 bits dentro del byte y los une en un string
    """
    conversion = "".join(f"{b:08b}" for b in datos_bytes)
    return conversion

Finalmente, definimos el mensaje arbitrario y utilizamos todas las funciones

In [12]:
m = input("Ingrese el mensaje: ").encode('utf-8')
#UTF-8 codifica caracteres en unicode

k = generar_clave(len(m))

e = otp(m,k)

m_descifrado = otp(e,k).decode('utf-8')

print(f"Mensaje (bits):       {bytes_a_bits(m)}")
print(f"Clave (bits):         {bytes_a_bits(k)}")

print("-"*200)

print(f"Criptograma (bits):   {bytes_a_bits(e)}")
print(f"\n Mensaje recuperado:   {m_descifrado}")

Ingrese el mensaje:  Hola


Mensaje (bits):       01001000011011110110110001100001
Clave (bits):         00000000000000010000000100000000
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Criptograma (bits):   01001000011011100110110101100001

 Mensaje recuperado:   Hola
